# Ghost-identity assertion-dose pilot -- free-GPU runner

Thin wrapper around `python -m ghost_identity.main`. All real logic lives in
the repo, not in this notebook, so the script and notebook paths never drift.

**Works on Kaggle (recommended) or Colab.** Before running:

| | Kaggle | Colab |
|---|---|---|
| Enable GPU | Settings -> Accelerator -> **GPU T4 x2** | Runtime -> Change runtime type -> **T4 GPU** |
| Enable internet | Settings -> **Internet on** (needs phone verification) | on by default |
| Persistent output | `/kaggle/working` (kept on commit) | mount Google Drive |
| Background run | **Save Version -> Save & Run All** survives closing the tab | tab must stay open |
| Free budget | ~30 GPU-hours/week, 12h/session | no guarantee, idles out ~90min |

The full sweep is roughly **1.5-2.5h** on one T4 with the default
`Qwen/Qwen2.5-0.5B`. Everything resumes, so a killed session just means
re-running the sweep cell.

In [ ]:
import os, subprocess, sys

ON_KAGGLE = os.path.exists("/kaggle/working")
BASE = "/kaggle/working" if ON_KAGGLE else "/content"
REPO_DIR = f"{BASE}/ghost-identity"
# /kaggle/working persists for the session and is saved with a committed
# version; on Colab point this at Drive if you want it to outlive the VM.
RUNS_DIR = f"{BASE}/runs"

print("kaggle" if ON_KAGGLE else "colab/other", "| repo:", REPO_DIR, "| runs:", RUNS_DIR)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "NO GPU -- enable the accelerator first")

## Get the code

**This repo is private**, so a plain `git clone` will 404. Pick one:

1. **Make the repo public** on GitHub -- then the token below is unnecessary.
2. **Use a read-only token** (what the cell below does): create a
   fine-grained PAT with *Contents: Read-only* on this repo, then store it as
   `GH_TOKEN` in Kaggle (Add-ons -> Secrets) or Colab (key icon -> Secrets).
3. **Upload the repo as a Kaggle Dataset** and skip cloning entirely.

In [ ]:
REPO_SLUG = "safiqsindha/ghost-identity"
BRANCH = "main"

token = None
try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GH_TOKEN")
    else:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
except Exception as e:
    print("No GH_TOKEN secret found -- assuming the repo is public.", e)

auth = f"x-access-token:{token}@" if token else ""
url = f"https://{auth}github.com/{REPO_SLUG}.git"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
# Never leave a token sitting in .git/config for the saved notebook output.
subprocess.run(["git", "remote", "set-url", "origin", f"https://github.com/{REPO_SLUG}.git"], check=False)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Kaggle/Colab already ship torch + transformers; peft is usually the only gap.
!pip install -q "peft>=0.10" "transformers>=4.40" "accelerate>=0.30"

import yaml
with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["paths"]["runs_dir"] = RUNS_DIR
with open("configs/notebook.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("model:", cfg["model"]["base_model_id"])
print("doses:", cfg["training"]["doses"], "seeds:", cfg["training"]["seeds"])
print("runs ->", cfg["paths"]["runs_dir"])

In [ ]:
# Sanity-check the whole path on CPU before spending GPU time (~2 seconds).
!python -m ghost_identity.main --dry-run --baseline --sweep --config configs/notebook.yaml

In [ ]:
# Baseline MUST run first, through the identical harness -- without a base
# rate nothing downstream is interpretable. Re-running is a fast no-op.
!python -m ghost_identity.main --baseline --config configs/notebook.yaml

In [ ]:
# The full 18-cell sweep (~1.5-2.5h on a T4). Safe to re-run after a killed
# session: finished cells are skipped via their .done markers, and training
# and eval have separate markers so a cell killed mid-eval doesn't retrain.
!python -m ghost_identity.main --sweep --config configs/notebook.yaml

In [ ]:
# Re-aggregate/re-plot from the saved raw completions, without regenerating.
!python -m ghost_identity.main --aggregate-only --config configs/notebook.yaml

import pandas as pd
from IPython.display import Image, display

display(Image(filename=f"{RUNS_DIR}/results/dose_response.png"))
pd.read_csv(f"{RUNS_DIR}/results/table.csv")